# Attention Head Importance

Score attention heads by importance using multiple methods: knockout effects,
output magnitude, logit attribution, composite ranking, and per-position analysis.

## Why This Matters

Attention head importance reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_importance import (
    head_knockout_importance,
    head_output_magnitude,
    head_logit_attribution,
    composite_importance_ranking,
    head_importance_by_position,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Head Knockout Importance

Zero each head's output and measure the effect on logits.

In [ ]:
result = head_knockout_importance(model, tokens)
print(f"Most important: L{result['most_important']['layer']}H{result['most_important']['head']} "
      f"(max_logit_change={result['most_important']['max_logit_change']:.4f})")
print(f"Least important: L{result['least_important']['layer']}H{result['least_important']['head']} "
      f"(max_logit_change={result['least_important']['max_logit_change']:.4f})")
print(f'\nAll heads by knockout impact:')
for h in result['per_head']:
    print(f"  L{h['layer']}H{h['head']}: max={h['max_logit_change']:.4f}, mean={h['mean_logit_change']:.4f}")

## Output Magnitude Ranking

Rank heads by the norm of their contribution to the residual stream.

In [ ]:
result = head_output_magnitude(model, tokens)
print('Heads ranked by output norm:')
for h in result['per_head']:
    bar = '█' * int(h['output_norm'] * 50)
    print(f"  L{h['layer']}H{h['head']}: {h['output_norm']:.4f} {bar}")

## Logit Attribution

Attribute logits for the predicted token to specific heads at a given position.

In [ ]:
result = head_logit_attribution(model, tokens, position=-1)
print(f"Target token: {result['target_token']} at position {result['position']}\n")
for h in result['per_head']:
    sign = '+' if h['logit_contribution'] > 0 else '-'
    print(f"  L{h['layer']}H{h['head']}: {sign}{h['abs_contribution']:.4f} "
          f"(raw: {h['logit_contribution']:+.4f})")

## Composite Importance Ranking

Combine output magnitude and logit attribution into a single composite score.

In [ ]:
result = composite_importance_ranking(model, tokens)
print(f"Most important: L{result['most_important']['layer']}H{result['most_important']['head']}\n")
for h in result['per_head']:
    print(f"  L{h['layer']}H{h['head']}: composite={h['composite_score']:.4f} "
          f"(norm={h['output_norm']:.4f}, logit_attr={h['logit_attribution']:.4f})")

## Per-Position Importance

See which head is most important at each token position.

In [ ]:
result = head_importance_by_position(model, tokens)
for pos in result['per_position']:
    top = pos['top_head']
    print(f"Position {pos['position']}: top head = L{top['layer']}H{top['head']} "
          f"(norm={top['output_norm']:.4f})")